# NBA Tracking Data — Season Stats Collection
*Run this notebook once to build `season_stats.csv` from the raw game archives.*

Processes all available games, computes per-player per-game stats, and saves to CSV.
The analysis notebook (`nba_tracking_experiments.ipynb`) loads from this file.

**Output columns:**

| Column | Description |
|---|---|
| `game_id` | NBA game ID |
| `player_id` / `player_name` / `team_abbr` | Player identity |
| `top_speed` | Highest smoothed speed in the game (ft/s) |
| `avg_speed` | Mean speed across all tracked frames (ft/s) |
| `dist_miles` | Total distance run (miles) |
| `hi_dist_miles` | Distance covered above sprint threshold (15 ft/s / ~10 mph) |
| `q1_avg_speed` / `q1_top_speed` / `q1_dist_miles` / `q1_hi_dist` | Q1 stats |
| `q4_avg_speed` / `q4_top_speed` / `q4_dist_miles` / `q4_hi_dist` | Q4 stats |
| `frames` | Total tracked frames (proxy for minutes: frames/25/60) |
| `frames_q1` / `frames_q4` | Frames in Q1/Q4 (proxy for quarter minutes) |

In [31]:
import json, os, glob, subprocess, tempfile
import numpy as np
import pandas as pd

DATA_DIR   = '/Users/willhanley/Desktop/spacing/nba-movement-data/data'
OUTPUT_CSV = '/Users/willhanley/Desktop/spacing/season_stats.csv'
FPS        = 25

archives = sorted(glob.glob(os.path.join(DATA_DIR, '*.7z')))
jsons    = sorted(glob.glob(os.path.join(DATA_DIR, '*.json')))
print(f'Archives: {len(archives)}')
print(f'Loose JSONs: {len(jsons)}')
print(f'Output: {OUTPUT_CSV}')

Archives: 636
Loose JSONs: 1
Output: /Users/willhanley/Desktop/spacing/season_stats.csv


## Processing Function

For each game:
1. Parse all moments into a flat DataFrame
2. Deduplicate (same moment appears in ~3 overlapping events)
3. Compute per-frame velocities and distances
4. Aggregate to one row per player with full-game and per-quarter stats

In [32]:
THRESHOLDS = [15, 18, 20]  # ft/s — multiple sprint thresholds, pick best in analysis

def process_game(filepath):
    try:
        with open(filepath) as f:
            game = json.load(f)
    except Exception:
        return []

    game_id = game.get('gameid', os.path.basename(filepath).replace('.json', ''))
    ev0     = game['events'][0]
    home    = ev0['home']
    visitor = ev0['visitor']
    home_id = home['teamid']
    vis_id  = visitor['teamid']

    player_meta = {}
    for p in home['players']:
        player_meta[p['playerid']] = {
            'name': f"{p['firstname']} {p['lastname']}",
            'team_id': home_id,
            'team_abbr': home['abbreviation'],
        }
    for p in visitor['players']:
        player_meta[p['playerid']] = {
            'name': f"{p['firstname']} {p['lastname']}",
            'team_id': vis_id,
            'team_abbr': visitor['abbreviation'],
        }

    rows = []
    for event in game['events']:
        for moment in event['moments']:
            quarter, game_clock = moment[0], moment[2]
            for player in moment[5]:
                pid = player[1]
                if pid == -1:
                    continue
                rows.append((quarter, game_clock, pid, player[2], player[3], event['eventId']))

    if not rows:
        return []

    df = pd.DataFrame(rows, columns=['quarter', 'game_clock', 'player_id', 'x', 'y', 'event_id'])

    df = (
        df.sort_values(['quarter', 'game_clock', 'player_id', 'event_id'],
                       ascending=[True, False, True, True])
          .drop_duplicates(subset=['quarter', 'game_clock', 'player_id'])
          .sort_values(['player_id', 'quarter', 'game_clock'],
                       ascending=[True, True, False])
          .reset_index(drop=True)
    )

    grp = df.groupby(['player_id', 'quarter'])
    df['dx']    = grp['x'].diff()
    df['dy']    = grp['y'].diff()
    df['speed'] = np.sqrt(df['dx']**2 + df['dy']**2) * FPS
    df['dist']  = np.sqrt(df['dx']**2 + df['dy']**2)

    df.loc[df['speed'] > 35, 'speed'] = np.nan
    df.loc[df['dist']  > 1.5, 'dist']  = np.nan

    df['speed_s'] = (
        df.groupby(['player_id', 'quarter'])['speed']
          .transform(lambda s: s.rolling(5, min_periods=1, center=True).mean())
    )

    results = []
    for pid, pdata in df.groupby('player_id'):
        if pid not in player_meta:
            continue
        meta = player_meta[pid]

        def _r(v):
            return round(float(v), 3) if (v is not None and not np.isnan(float(v))) else np.nan

        def q_stats(q):
            qd = pdata[pdata['quarter'] == q]
            if qd.empty:
                return dict(avg_speed=np.nan, top_speed=np.nan,
                            dist_miles=np.nan, frames=0,
                            **{f'hi_dist_{t}': np.nan for t in THRESHOLDS})
            d = dict(
                avg_speed  = float(qd['speed_s'].mean()),
                top_speed  = float(qd['speed_s'].max()),
                dist_miles = float(qd['dist'].sum()) / 5280,
                frames     = len(qd),
            )
            for t in THRESHOLDS:
                d[f'hi_dist_{t}'] = float(qd.loc[qd['speed_s'] > t, 'dist'].sum()) / 5280
            return d

        q1, q4 = q_stats(1), q_stats(4)

        row = {
            'game_id':     game_id,
            'player_id':   pid,
            'player_name': meta['name'],
            'team_id':     meta['team_id'],
            'team_abbr':   meta['team_abbr'],
            'top_speed':   _r(pdata['speed_s'].max()),
            'avg_speed':   _r(pdata['speed_s'].mean()),
            'dist_miles':  _r(pdata['dist'].sum() / 5280),
            'frames':      len(pdata),
            # per-threshold full-game hi_dist
            **{f'hi_dist_{t}': _r(pdata.loc[pdata['speed_s'] > t, 'dist'].sum() / 5280)
               for t in THRESHOLDS},
            # Q1
            'q1_avg_speed':  _r(q1['avg_speed']),
            'q1_top_speed':  _r(q1['top_speed']),
            'q1_dist_miles': _r(q1['dist_miles']),
            'frames_q1':     q1['frames'],
            **{f'q1_hi_{t}': _r(q1[f'hi_dist_{t}']) for t in THRESHOLDS},
            # Q4
            'q4_avg_speed':  _r(q4['avg_speed']),
            'q4_top_speed':  _r(q4['top_speed']),
            'q4_dist_miles': _r(q4['dist_miles']),
            'frames_q4':     q4['frames'],
            **{f'q4_hi_{t}': _r(q4[f'hi_dist_{t}']) for t in THRESHOLDS},
        }
        results.append(row)

    return results


def extract_and_process(archive_path, tmpdir):
    r = subprocess.run(
        ['7z', 'e', archive_path, f'-o{tmpdir}', '-y'],
        capture_output=True
    )
    if r.returncode != 0:
        return []
    rows = []
    for j in glob.glob(os.path.join(tmpdir, '*.json')):
        rows.extend(process_game(j))
        os.remove(j)
    return rows

print(f'Functions defined.  Thresholds: {THRESHOLDS} ft/s')

Functions defined.  Thresholds: [15, 18, 20] ft/s


## Run — Process All Games

Extracts each `.7z` one at a time, processes it, deletes the temp JSON, moves on.
Progress printed every 50 games. **~45-60 min for all 636 games.**

Skips if `season_stats.csv` already exists — delete the file to reprocess.

In [33]:
if os.path.exists(OUTPUT_CSV):
    print(f'season_stats.csv already exists — skipping.')
    print(f'Delete {OUTPUT_CSV} and re-run to reprocess.')
else:
    all_rows = []

    # Process any loose JSONs first
    for j in jsons:
        all_rows.extend(process_game(j))
        print(f'  Processed loose JSON: {os.path.basename(j)} ({len(all_rows)} rows so far)')

    # Process archives
    with tempfile.TemporaryDirectory() as tmpdir:
        total = len(archives)
        for i, archive in enumerate(archives):
            all_rows.extend(extract_and_process(archive, tmpdir))
            if (i + 1) % 50 == 0 or (i + 1) == total:
                print(f'  {i+1:3d}/{total}  |  {len(all_rows):,} player-game rows so far')

    df_out = pd.DataFrame(all_rows)
    df_out.to_csv(OUTPUT_CSV, index=False)
    print(f'\nSaved {len(df_out):,} rows → {OUTPUT_CSV}')
    print(f'Columns: {list(df_out.columns)}')

season_stats.csv already exists — skipping.
Delete /Users/willhanley/Desktop/spacing/season_stats.csv and re-run to reprocess.


## Verify Output

Quick sanity check once the file is saved.

## Build Final Player Dataset

Aggregates the per-game rows into one row per player with all metrics pre-computed:
- **`hi_dist_per_36`** — sprint miles per 36 min (energy metric, usage-adjusted)
- **`hi_dist_fatigue_pct`** — % drop in sprint activity from Q1 to Q4 (fatigue metric)
- Fatigue computed only from games where player logged 5+ min in both Q1 and Q4

In [34]:
df = pd.read_csv(OUTPUT_CSV)

print(f'Rows:    {len(df):,}')
print(f'Games:   {df["game_id"].nunique()}')
print(f'Players: {df["player_id"].nunique()}')
print(f'Columns: {list(df.columns)}')
print()

df['minutes'] = df['frames'] / 25 / 60
full_game = df[df['frames'] >= 300]
print(f'Player-games with 12+ min tracked: {len(full_game):,}')

has_q1 = (df['frames_q1'] >= 125).sum()
has_q4 = (df['frames_q4'] >= 125).sum()
both   = ((df['frames_q1'] >= 125) & (df['frames_q4'] >= 125)).sum()
print(f'Player-games with 5+ min in Q1:   {has_q1:,}')
print(f'Player-games with 5+ min in Q4:   {has_q4:,}')
print(f'Player-games with 5+ min in both: {both:,}')
print()

for t in THRESHOLDS:
    col = f'hi_dist_{t}'
    print(f'Threshold {t} ft/s — avg per game: {full_game[col].mean():.3f} mi  '
          f'| max: {full_game[col].max():.3f} mi ({full_game.loc[full_game[col].idxmax(), "player_name"]})')

print()
print('Sample row:')
print(full_game.iloc[0][['player_name','team_abbr','avg_speed','dist_miles',
                          'hi_dist_15','hi_dist_18','hi_dist_20',
                          'frames','frames_q1','frames_q4']].to_string())

Rows:    13,495
Games:   631
Players: 447
Columns: ['game_id', 'player_id', 'player_name', 'team_id', 'team_abbr', 'top_speed', 'avg_speed', 'dist_miles', 'frames', 'hi_dist_15', 'hi_dist_18', 'hi_dist_20', 'q1_avg_speed', 'q1_top_speed', 'q1_dist_miles', 'frames_q1', 'q1_hi_15', 'q1_hi_18', 'q1_hi_20', 'q4_avg_speed', 'q4_top_speed', 'q4_dist_miles', 'frames_q4', 'q4_hi_15', 'q4_hi_18', 'q4_hi_20']

Player-games with 12+ min tracked: 13,464
Player-games with 5+ min in Q1:   11,398
Player-games with 5+ min in Q4:   11,244
Player-games with 5+ min in both: 9,472

Threshold 15 ft/s — avg per game: 0.200 mi  | max: 0.715 mi (Andre Roberson)
Threshold 18 ft/s — avg per game: 0.057 mi  | max: 0.316 mi (Andre Roberson)
Threshold 20 ft/s — avg per game: 0.020 mi  | max: 0.151 mi (Andre Roberson)

Sample row:
player_name    Luis Scola
team_abbr             TOR
avg_speed           6.463
dist_miles           1.51
hi_dist_15          0.232
hi_dist_18          0.067
hi_dist_20          0.013
frame

In [35]:
PLAYER_CSV    = '/Users/willhanley/Desktop/spacing/player_stats.csv'
MIN_GAMES     = 20
MIN_FAT_GAMES = 15

raw = pd.read_csv(OUTPUT_CSV)
raw['minutes'] = raw['frames'] / 25 / 60

full = raw[raw['frames'] >= 300].copy()

# Build hi_dist agg columns for all thresholds
hi_dist_agg = {f'total_hi_{t}': (f'hi_dist_{t}', 'sum') for t in THRESHOLDS}

base = (
    full.groupby(['player_id', 'player_name', 'team_abbr'])
    .agg(
        games         = ('game_id',   'nunique'),
        avg_minutes   = ('minutes',   'mean'),
        total_minutes = ('minutes',   'sum'),
        avg_speed     = ('avg_speed', 'mean'),
        top_speed     = ('top_speed', 'max'),
        total_miles   = ('dist_miles','sum'),
        **hi_dist_agg,
    )
    .reset_index()
    .query(f'games >= {MIN_GAMES}')
    .reset_index(drop=True)
)

# hi_dist_per_36 for each threshold
for t in THRESHOLDS:
    base[f'hi_dist_{t}_per_36'] = base[f'total_hi_{t}'] / base['total_minutes'] * 36

# Fatigue: games with 5+ min in both Q1 and Q4
fat_games = raw[(raw['frames_q1'] >= 125) & (raw['frames_q4'] >= 125)].copy()

fat_agg = {'avg_q1_speed': ('q1_avg_speed', 'mean'), 'avg_q4_speed': ('q4_avg_speed', 'mean')}
for t in THRESHOLDS:
    fat_agg[f'avg_q1_hi_{t}'] = (f'q1_hi_{t}', 'mean')
    fat_agg[f'avg_q4_hi_{t}'] = (f'q4_hi_{t}', 'mean')

fatigue = (
    fat_games.groupby('player_id')
    .agg(fatigue_games=('game_id', 'nunique'), **fat_agg)
    .reset_index()
    .query(f'fatigue_games >= {MIN_FAT_GAMES}')
    .reset_index(drop=True)
)

# Q1→Q4 sprint drop for each threshold
for t in THRESHOLDS:
    fatigue[f'fatigue_pct_{t}'] = (
        (fatigue[f'avg_q1_hi_{t}'] - fatigue[f'avg_q4_hi_{t}'])
        / fatigue[f'avg_q1_hi_{t}'] * 100
    )
fatigue['speed_fatigue_pct'] = (
    (fatigue['avg_q1_speed'] - fatigue['avg_q4_speed'])
    / fatigue['avg_q1_speed'] * 100
)

players = base.merge(fatigue, on='player_id', how='left')
players.to_csv(PLAYER_CSV, index=False)
print(f'Saved {len(players):,} players → {PLAYER_CSV}')
print(f'Columns: {[c for c in players.columns]}')
print()

# Preview leaderboard at each threshold
for t in THRESHOLDS:
    col = f'hi_dist_{t}_per_36'
    print(f'--- Top 10 by hi_dist_per_36 at {t} ft/s threshold ---')
    print(players.nlargest(10, col)[['player_name','team_abbr','avg_minutes', col]].to_string(index=False))
    print()

Saved 349 players → /Users/willhanley/Desktop/spacing/player_stats.csv
Columns: ['player_id', 'player_name', 'team_abbr', 'games', 'avg_minutes', 'total_minutes', 'avg_speed', 'top_speed', 'total_miles', 'total_hi_15', 'total_hi_18', 'total_hi_20', 'hi_dist_15_per_36', 'hi_dist_18_per_36', 'hi_dist_20_per_36', 'fatigue_games', 'avg_q1_speed', 'avg_q4_speed', 'avg_q1_hi_15', 'avg_q4_hi_15', 'avg_q1_hi_18', 'avg_q4_hi_18', 'avg_q1_hi_20', 'avg_q4_hi_20', 'fatigue_pct_15', 'fatigue_pct_18', 'fatigue_pct_20', 'speed_fatigue_pct']

--- Top 10 by hi_dist_per_36 at 15 ft/s threshold ---
     player_name team_abbr  avg_minutes  hi_dist_15_per_36
  Andre Roberson       OKC    20.930341           0.655237
    Jeremy Evans       DAL     8.657394           0.636030
    Kyle Singler       OKC    10.815232           0.630626
Dante Cunningham       NOP    18.610483           0.627953
  Doug McDermott       CHI    19.749404           0.604032
     Cody Zeller       CHA    24.386496           0.585456
